# L76 · 音乐理论速成 — 音高、音程、色度（chroma）轮（chroma wheel）与十二平均律

**学习目标**
- 理解 12 个音高类别（pitch class）和八度等价性（octave equivalence）
- 掌握 MIDI ↔ 频率转换公式
- 理解音程（interval）：两音之间的半音距离及常见音程名称
- 实现色度轮可视化
- 明白为什么 audio AI 用 chroma 而非原始频率


## 音乐的基本单元：音高类别

人耳把频率每翻一倍（一个八度）感知为"同名音"：
130Hz C2、261Hz C3、523Hz C4 在和声上是等价的。

**十二平均律（12-EDO）** 把一个八度等分成 12 个半音（semitone）：

| 编号 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 音名 | C | C# | D | D# | E | F | F# | G | G# | A | A# | B |

**MIDI 公式**：`f(n) = 440 × 2^((n-69)/12)`，其中 n 是 MIDI 编号（A4=440Hz=MIDI69）

**色度 = MIDI mod 12**：忽略八度，只保留音高类别。这是音乐 AI 的核心特征。

## 音程（Interval）：两音之间的距离

**音程** = 两个音之间相差的**半音（semitone）数**，在 MIDI 空间里就是两个编号之差的绝对值。

| 音程名称 | 半音数 | 示例（从 C 起算） |
|---|:---:|---|
| 小二度 minor 2nd | 1 | C → C# |
| 大二度 major 2nd | 2 | C → D |
| 小三度 minor 3rd | 3 | C → D# |
| 大三度 major 3rd | 4 | C → E |
| 纯四度 perfect 4th | 5 | C → F |
| 增四度 / 三全音 tritone | 6 | C → F# |
| 纯五度 perfect 5th | 7 | C → G |
| 大六度 major 6th | 9 | C → A |
| 纯八度 octave | 12 | C → C（高八度）|

**MIDI 音程公式**：`interval_semitones = |MIDI_高 - MIDI_低|`

例：C4（MIDI 60）到 G4（MIDI 67）= 7 个半音 = 纯五度。

> **思考题**：A3（MIDI 57）到 E4（MIDI 64）相差几个半音？这是什么音程？  
> （答：|64-57|=7 半音 = 纯五度）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PITCH_CLASSES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
MIDI_A4, FREQ_A4 = 69, 440.0

## ✏️ 任务 1：MIDI ↔ 频率转换

In [ ]:
def midi_to_freq(midi: float) -> float:
    """MIDI 编号 → 频率 Hz。A4=69=440Hz。"""
    # ✏️ TODO: f(n) = 440 × 2^((n-69)/12)
    raise NotImplementedError("TODO")

def freq_to_midi(freq: float) -> float:
    """频率 Hz → 浮点 MIDI 编号。"""
    # ✏️ TODO: 逆公式：n = 69 + 12 × log2(f / 440)
    raise NotImplementedError("TODO")

# 验证
for name, midi, expected in [("A4",69,440.0),("C4",60,261.63),("A3",57,220.0)]:
    got = midi_to_freq(midi)
    ok = abs(got - expected) < 0.5
    print(f"{name} MIDI{midi}: {got:.2f} Hz  {chr(9989) if ok else chr(10060)}")
    assert ok, f"{name} MIDI{midi}: 期望 {expected:.2f} Hz，实际 {got:.2f} Hz — 检查公式 f(n)=440×2^((n-69)/12)"

assert abs(freq_to_midi(440.0) - 69) < 0.01, "freq_to_midi(440) 应返回 69"
assert abs(midi_to_freq(0) - 8.18) < 0.1, "MIDI 0（最低音）应约为 8.18 Hz"
print("✅ 频率转换验证通过")


## ✏️ 任务 2：chroma_from_freq

In [ ]:
def chroma_from_freq(freq: float) -> int:
    """频率 Hz → 音高类别 0-11 (C=0, C#=1, ..., B=11)。"""
    # ✏️ TODO: 用 freq_to_midi 换算后 round() 取整，再取 mod 12
    # ⚠️ 必须用 round() 而非 int()（截断）：
    #   B4=493.88Hz → freq_to_midi ≈ 70.9999
    #   int(70.9999)=70 → A# (错误)，round(70.9999)=71 → B (正确) ✓
    # 参考实现：n = round(freq_to_midi(freq)); return n % 12
    raise NotImplementedError("TODO")

# 验证（含 B4 取整边界用例）
for name, freq, expected_pc in [("C4",261.63,0),("A4",440.0,9),("A5",880.0,9),("B4",493.88,11)]:
    pc = chroma_from_freq(freq)
    print(f"{name} ({freq:.1f}Hz): chroma={pc} ({PITCH_CLASSES[pc]}) {chr(9989) if pc==expected_pc else chr(10060)}")
    assert pc == expected_pc, (
        f"{name} chroma 错误：期望 {expected_pc} ({PITCH_CLASSES[expected_pc]})，"
        f"实际 {pc} — 确认用了 round() 而非 int()"
    )

print("✅ chroma 验证通过（含 B4 取整边界）")


In [ ]:
from pathlib import Path
# 🎨 色度轮可视化（教学图表，aviz 不涵盖此内容）
fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'projection': 'polar'})
angles = np.linspace(0, 2*np.pi, 12, endpoint=False)
colors = plt.cm.hsv(np.linspace(0, 1, 12, endpoint=False))
ax.bar(angles, np.ones(12), width=2*np.pi/12, color=colors, alpha=0.75, align='edge')
for i, (a, name) in enumerate(zip(angles + np.pi/12, PITCH_CLASSES)):
    ax.text(a, 1.35, name, ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.set_title('色度轮 — 12 音高类别', pad=15)
plt.tight_layout()
plt.savefig(Path.cwd() / 'chroma_wheel.png', dpi=80, bbox_inches='tight')
plt.show()

## 大调音阶与 chroma

C 大调 = C-D-E-F-G-A-B = chroma {0,2,4,5,7,9,11}。

同一首曲子升调后转调（transposition），整个 chroma 向量旋转。
不同调性（tonality）的歌曲，chroma 分布形状不同但"模式"相似。

这就是为什么 chroma 向量能做**调性无关的**音乐相似度比较。

## 本课收束

| 概念 | 要记住 |
|---|---|
| 12-EDO | 八度=12半音，相邻半音频率比 = 2^(1/12) ≈ 1.0595 |
| MIDI | f(n)=440×2^((n-69)/12)，A4=MIDI69=440Hz |
| Chroma | MIDI mod 12，消除八度，保留音高类别 |
| L77 | 把 chroma 做到 STFT 上，得到时变 chromagram，调用 aurora.music |

下一课（L77）用 Aurora music 模块提取色度（chroma）、RMS、ZCR 等特征，把音乐理论转换成可计算的数值向量。